# Study 2: Most Common Taxi Trips


## Setup


In [3]:
import sys
import pyspark
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
from pyspark.sql.functions import *
import time
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np
import pandas as pd

spark = SparkSession.builder.master('local[*]').getOrCreate()

sc = spark.sparkContext
spark

26/10/20 14:41:47 WARN Utils: Your hostname, Andres-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.37 instead (on interface en0)
26/10/20 14:41:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/20 14:41:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Data Loading and Cleaning


In [6]:
df = spark.read.format("parquet").option("inferSchema", "true").option("timestampFormat","yyyy-MM-dd HH:mm:ss").option("header", "true").option("mode", "DROPMALFORMED").load("yellow_tripdata_2025-01.parquet")
df = df.filter((df["passenger_count"] > 0) & (df["trip_distance"] > 0) & (df["fare_amount"] > 0) & (df["tip_amount"] >= 0))
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

## RDD


In [ ]:
import time
from pyspark.sql import functions as F

#Load file and clean it by removing unavailable names
taxi_zones = spark.read.format("csv").option("inferSchema", "true").option("header", "true").load("taxi_zone_lookup.csv")
taxi_zones = taxi_zones.select("LocationID", "Zone").filter(F.col("LocationID").isNotNull() & F.col("Zone").isNotNull() & (F.trim(F.col("Zone")) != "N/A"))

trip_record_count = df.count()
zone_record_count = taxi_zones.count()

start_time = time.perf_counter()

trip_records = df.select("PULocationID", "DOLocationID").rdd
zone_records = taxi_zones.rdd

#Map each location ID to its name and each pickup/dropoff pair to a count of 1
zone_names = zone_records.map(lambda row: (row["LocationID"], row["Zone"]))
trip_pairs = trip_records.map(lambda row: ((row["PULocationID"], row["DOLocationID"]), 1))

#Count the journeys for each pickup/dropoff pair 
route_counts = trip_pairs.reduceByKey(lambda first_count, second_count: first_count + second_count)

#Use the pickup ID as the key to join the pickup zone name
pickup_routes = route_counts.map(lambda route: (route[0][0], (route[0][1], route[1]))).join(zone_names)

#Switch the key to the dropoff ID, join its name and simplify the result
named_routes = pickup_routes.map(lambda route: (route[1][0][0], (route[1][1], route[1][0][1]))).join(zone_names)
named_routes = named_routes.map(lambda route: (route[1][0][0], route[1][1], route[1][0][1]))

#Convert to a DataFrame and display all routes from most to least popular
popular_routes_df = named_routes.toDF(["Pickup Zone", "Dropoff Zone", "Trip Count"])
popular_routes_df = popular_routes_df.orderBy(F.desc("Trip Count"), "Pickup Zone", "Dropoff Zone")
popular_routes_df.show(n=popular_routes_df.count(), truncate=False)

#Calculate execution time and throughput
execution_time = time.perf_counter() - start_time
total_input_records = trip_record_count + zone_record_count
processing_speed = total_input_records / execution_time

#Display the results
print("--- Most Common Taxi Trips: RDD Performance ---")
print(f"Execution Time: {execution_time:.4f} seconds")
print(f"Total Input Records: {total_input_records:,}")
print(f"Processing Speed: {processing_speed:,.2f} rows/second")


+-----------------------------------+---------------------------------------------+----------+
|Pickup Zone                        |Dropoff Zone                                 |Trip Count|
+-----------------------------------+---------------------------------------------+----------+
|Upper East Side South              |Upper East Side North                        |23456     |
|Upper East Side North              |Upper East Side South                        |20262     |
|Upper East Side North              |Upper East Side North                        |16401     |
|Upper East Side South              |Upper East Side South                        |15631     |
|Midtown Center                     |Upper East Side South                        |10711     |
|Upper East Side South              |Midtown Center                               |9367      |
|Midtown Center                     |Upper East Side North                        |9153      |
|Upper West Side South              |Upper West Si

## DataFrame


In [ ]:
import time
from pyspark.sql import functions as F

#Count the input records separately from the execution time
trip_record_count = df.count()
zone_record_count = taxi_zones.count()

start_time_df = time.perf_counter()

#Count the journeys for each pickup/dropoff pair
route_counts_df = df.groupBy("PULocationID", "DOLocationID").agg(F.count("*").alias("Trip Count"))

#Prepare the lookup columns for the pickup and dropoff joins
pickup_zones_df = taxi_zones.select(F.col("LocationID").alias("PULocationID"), F.col("Zone").alias("Pickup Zone"))
dropoff_zones_df = taxi_zones.select(F.col("LocationID").alias("DOLocationID"), F.col("Zone").alias("Dropoff Zone"))

#Join the pickup and dropoff zone names using their location IDs
pickup_routes_df = route_counts_df.join(pickup_zones_df, "PULocationID", "inner")
named_routes_df = pickup_routes_df.join(dropoff_zones_df, "DOLocationID", "inner")

#Select the same output columns as the RDD version and sort by popularity
popular_routes_dataframe = named_routes_df.select("Pickup Zone", "Dropoff Zone", "Trip Count")
popular_routes_dataframe = popular_routes_dataframe.orderBy(F.desc("Trip Count"), "Pickup Zone", "Dropoff Zone")
popular_routes_dataframe.show(n=popular_routes_dataframe.count(), truncate=False)

#Calculate execution time and throughput
execution_time_df = time.perf_counter() - start_time_df
total_input_records = trip_record_count + zone_record_count
processing_speed_df = total_input_records / execution_time_df

#Display the results
print("--- Most Common Taxi Trips: DataFrame Performance ---")
print(f"Execution Time: {execution_time_df:.4f} seconds")
print(f"Total Input Records: {total_input_records:,}")
print(f"Processing Speed: {processing_speed_df:,.2f} rows/second")


+-----------------------------------+---------------------------------------------+----------+
|Pickup Zone                        |Dropoff Zone                                 |Trip Count|
+-----------------------------------+---------------------------------------------+----------+
|Upper East Side South              |Upper East Side North                        |23456     |
|Upper East Side North              |Upper East Side South                        |20262     |
|Upper East Side North              |Upper East Side North                        |16401     |
|Upper East Side South              |Upper East Side South                        |15631     |
|Midtown Center                     |Upper East Side South                        |10711     |
|Upper East Side South              |Midtown Center                               |9367      |
|Midtown Center                     |Upper East Side North                        |9153      |
|Upper West Side South              |Upper West Si

## SQL


In [14]:
import time

#Prepare data for querying the trips and cleaned zone lookup
df.createOrReplaceTempView("trips")
taxi_zones.createOrReplaceTempView("zones")

trip_record_count = df.count()
zone_record_count = taxi_zones.count()

start_time_sql = time.perf_counter()

#Count each pickup/dropoff pair, attach the zone names and sort by popularity
query = """
WITH route_counts AS (
    SELECT PULocationID, DOLocationID, COUNT(*) AS trip_count
    FROM trips
    GROUP BY PULocationID, DOLocationID
)
SELECT pickup_zones.Zone AS `Pickup Zone`,
       dropoff_zones.Zone AS `Dropoff Zone`,
       route_counts.trip_count AS `Trip Count`
FROM route_counts
INNER JOIN zones AS pickup_zones ON route_counts.PULocationID = pickup_zones.LocationID
INNER JOIN zones AS dropoff_zones ON route_counts.DOLocationID = dropoff_zones.LocationID
ORDER BY `Trip Count` DESC, `Pickup Zone`, `Dropoff Zone`
"""

#Execute the query and display all matched routes
popular_routes_sql = spark.sql(query)
popular_routes_sql.show(n=popular_routes_sql.count(), truncate=False)

#Calculate execution time and throughput, including counting and displaying the results
execution_time_sql = time.perf_counter() - start_time_sql
total_input_records = trip_record_count + zone_record_count
processing_speed_sql = total_input_records / execution_time_sql

print("--- Most Common Taxi Trips: SQL Performance ---")
print(f"Execution Time: {execution_time_sql:.4f} seconds")
print(f"Total Input Records: {total_input_records:,}")
print(f"Processing Speed: {processing_speed_sql:,.2f} rows/second")


+-----------------------------------+---------------------------------------------+----------+
|Pickup Zone                        |Dropoff Zone                                 |Trip Count|
+-----------------------------------+---------------------------------------------+----------+
|Upper East Side South              |Upper East Side North                        |23456     |
|Upper East Side North              |Upper East Side South                        |20262     |
|Upper East Side North              |Upper East Side North                        |16401     |
|Upper East Side South              |Upper East Side South                        |15631     |
|Midtown Center                     |Upper East Side South                        |10711     |
|Upper East Side South              |Midtown Center                               |9367      |
|Midtown Center                     |Upper East Side North                        |9153      |
|Upper West Side South              |Upper West Si